# Individual Porfolio
-------------------------------------------------------------------------------------------------------------------------------------------------------------

## Basketball Game
-------------------------------------------------------------------------------------------------------------------------------------------------------------

## Basketball Evasion — CS111 Code Review

I built a basketball evasion game where you dodge LeBron James on a court. Below I break down every CS111 concept using real code from the game.

---

## Control Structures

### Iteration — `for`, `forEach`, `while` loops

The court's wood plank lines use a `for` loop:
```javascript
for (let x = 0; x < W; x += W/28) {
    ctx.beginPath(); ctx.moveTo(x, 0); ctx.lineTo(x, H); ctx.stroke();
}
```

The bench legs use `forEach` to draw 3 positions without repeating code:
```javascript
[.1, .5, .82].forEach(fx => 
    ctx.fillRect(o.x + o.w*fx, o.y + o.h*.4, o.w*.1, o.h*.55)
);
```

---

### Conditionals — `if/else`

Only move if the path isn't blocked:
```javascript
if (!blocked(p.x + dx, p.y, PR)) p.x += dx;
if (!blocked(p.x, p.y + dy, PR)) p.y += dy;
```

LeBron tries direct path first, then slides along walls:
```javascript
if (!blocked(l.x+mx, l.y+my, LR)) { l.x += mx; l.y += my; }
else {
    if (!blocked(l.x+mx, l.y, LR)) l.x += mx;
    if (!blocked(l.x, l.y+my, LR)) l.y += my;
}
```

---

### Nested Conditions — multi-level conditionals

Catch check with nested high score update:
```javascript
if (dist < PR + LR + 2) {
    over = true;
    if (elapsed > best) best = elapsed;
}
```

Speed warning with nested opacity calculation:
```javascript
if (spd > 2.8) {
    const alpha = Math.min((spd - 2.8) * 0.5, 0.9);
    ctx.fillStyle = `rgba(230,50,50,${alpha})`;
    ctx.fillText('⚡ LeBron is heating up!', W/2, 54);
}
```

---

## Data Types

### Numbers — position, velocity, score tracking
```javascript
const PR = 12;    // player radius
const LR = 15;    // LeBron radius
let elapsed = 0;  // time survived
let best = 0;     // best time
let tick = 0;     // frame counter
let dx = 0, dy = 0;  // velocity
```

---

### Strings — character names, game states, template literals
```javascript
ctx.fillText('11', 0, -28);   // player jersey number string
ctx.fillText('23', 0, -28);   // LeBron jersey number string
ctx.fillText(`⏱ ${elapsed.toFixed(1)}s`, W/2, 24);  // template literal
keys[e.key.toLowerCase()] = true;  // string key name
```

---

### Booleans — flags like `isOver`, `isPaused`
```javascript
let over = false;           // boolean — is game over?
if (over) return;           // gate all logic
if (elapsed > best) best = elapsed;
if (e.key.toLowerCase() === 'r' && over) reset();  // && AND
```

---

### Arrays — game object collections, level data
```javascript
const OBS = [
    { x: W*.25, y: H*.25, w: W*.12, h: H*.08 },
    { x: W*.55, y: H*.55, w: W*.12, h: H*.08 },
    // boundary walls...
];
[.1, .5, .82].forEach(fx => ctx.fillRect(...));
['arrowup','arrowdown','arrowleft','arrowright'].includes(e.key.toLowerCase())
```

---

### Objects (JSON) — configuration objects, sprite data
```javascript
let p = { x: W*.08, y: H*.5, dir: 1 };   // player object
let l = { x: W*.90, y: H*.5, dir: -1 };  // LeBron object
let keys = {};  // keyboard state map
{ x: W*.25, y: H*.25, w: W*.12, h: H*.08 }  // obstacle config object
```

---

## Operators

### Mathematical — physics calculations
```javascript
const dist = Math.sqrt(ddx*ddx + ddy*ddy);  // distance formula
const nx = ddx / dist;   // normalize direction
const mx = nx * spd;     // scale by speed
if (dx && dy) { dx *= 0.707; dy *= 0.707; }  // diagonal fix
const spd = Math.min(1.5 + elapsed * 0.04, 4);  // speed over time
const bounce = Math.sin(tick * 0.22) * 5;  // sine wave bounce
```

---

### String Operations — template literals, concatenation
```javascript
ctx.fillText(`⏱ ${elapsed.toFixed(1)}s`, W/2, 24);
ctx.fillText(`Best: ${best.toFixed(1)}s`, W/2+145, 24);
ctx.fillText(`Survived  ${elapsed.toFixed(1)}s`, W/2, by+105);
keys[e.key.toLowerCase()] = true;  // .toLowerCase() string method
```

---

### Boolean Expressions — `&&`, `||`, `!`
```javascript
if (keys['w'] || keys['arrowup']) dy = -3.5;   // OR
if (keys['a'] || keys['arrowleft']) dx = -3.5;  // OR
if (!blocked(p.x+dx, p.y, PR)) p.x += dx;       // NOT
if (e.key.toLowerCase() === 'r' && over) reset(); // AND
if (dx && dy) { dx *= .707; dy *= .707; }         // AND
```

---

## Play the Game

%%html
<div style="width:100%;background:#111;border-radius:8px;overflow:hidden;margin:20px 0;">
<canvas id="bball-cs111" style="display:block;width:100%;"></canvas>
<p style="font-family:monospace;font-size:12px;color:#888;padding:8px 12px;margin:0;background:#1a1a1a;">
Click court to focus · WASD / Arrow Keys to move · R to restart
</p>
</div>
<script>
(function(){
const canvas=document.getElementById('bball-cs111');
if(!canvas)return;
const W=canvas.width=canvas.offsetWidth||700;
const H=canvas.height=Math.round(W*.52);
canvas.style.height=H+'px';
canvas.setAttribute('tabindex','0');
const ctx=canvas.getContext('2d');
const OBS=[
  {x:W*.25,y:H*.25,w:W*.12,h:H*.08},{x:W*.55,y:H*.55,w:W*.12,h:H*.08},
  {x:W*.35,y:H*.60,w:W*.10,h:H*.08},{x:W*.60,y:H*.20,w:W*.10,h:H*.08},
  {x:0,y:0,w:4,h:H},{x:W-4,y:0,w:4,h:H},{x:0,y:0,w:W,h:4},{x:0,y:H-4,w:W,h:4},
];
const PR=12,LR=15;
let p,l,keys,over,t0,elapsed,best=0,tick;
const reset=()=>{
  p={x:W*.08,y:H*.5,dir:1};l={x:W*.90,y:H*.5,dir:-1};
  keys={};over=false;t0=Date.now();elapsed=0;tick=0;
};
reset();
canvas.addEventListener('keydown',e=>{
  keys[e.key.toLowerCase()]=true;
  if(['arrowup','arrowdown','arrowleft','arrowright'].includes(e.key.toLowerCase()))e.preventDefault();
});
canvas.addEventListener('keyup',e=>{
  keys[e.key.toLowerCase()]=false;
  if(e.key.toLowerCase()==='r'&&over)reset();
});
canvas.addEventListener('click',()=>canvas.focus());
const blocked=(cx,cy,r)=>OBS.some(o=>{
  const nx=Math.max(o.x,Math.min(cx,o.x+o.w));
  const ny=Math.max(o.y,Math.min(cy,o.y+o.h));
  return (cx-nx)**2+(cy-ny)**2<r*r;
});
const drawCourt=()=>{
  const g=ctx.createLinearGradient(0,0,W,0);
  g.addColorStop(0,'#b5651d');g.addColorStop(.5,'#cd853f');g.addColorStop(1,'#b5651d');
  ctx.fillStyle=g;ctx.fillRect(0,0,W,H);
  ctx.strokeStyle='rgba(90,40,0,.2)';ctx.lineWidth=1;
  for(let x=0;x<W;x+=W/28){ctx.beginPath();ctx.moveTo(x,0);ctx.lineTo(x,H);ctx.stroke();}
  ctx.strokeStyle='rgba(255,255,255,.7)';ctx.lineWidth=2;
  const pad=16;
  ctx.strokeRect(pad,pad,W-pad*2,H-pad*2);
  ctx.beginPath();ctx.moveTo(W/2,pad);ctx.lineTo(W/2,H-pad);ctx.stroke();
  ctx.beginPath();ctx.arc(W/2,H/2,H*.13,0,Math.PI*2);ctx.stroke();
};
const drawObs=()=>OBS.slice(0,4).forEach(o=>{
  ctx.fillStyle='#7a5230';ctx.fillRect(o.x,o.y,o.w,o.h*.4);
  ctx.fillStyle='#4a2f10';
  [.1,.5,.82].forEach(fx=>ctx.fillRect(o.x+o.w*fx,o.y+o.h*.4,o.w*.1,o.h*.55));
});
const drawFigure=(x,y,jersey,num,dir)=>{
  ctx.save();ctx.translate(Math.round(x),Math.round(y));
  ctx.fillStyle='rgba(0,0,0,.22)';
  ctx.beginPath();ctx.ellipse(0,4,10,3,0,0,Math.PI*2);ctx.fill();
  ctx.fillStyle='#111';ctx.fillRect(-9,-5,8,5);ctx.fillRect(1,-5,8,5);
  ctx.fillStyle=jersey;ctx.fillRect(-9,-20,18,15);ctx.fillRect(-10,-38,20,18);
  ctx.fillStyle='#c8784a';ctx.fillRect(-15,-37,5,13);ctx.fillRect(10,-37,5,13);
  ctx.fillStyle='#fff';ctx.font='bold 9px monospace';
  ctx.textAlign='center';ctx.textBaseline='middle';ctx.fillText(num,0,-28);
  ctx.fillStyle='#c8784a';ctx.fillRect(-3,-41,6,4);
  ctx.beginPath();ctx.arc(0,-50,9,0,Math.PI*2);ctx.fill();
  ctx.fillStyle='#111';ctx.fillRect(dir*2,-52,3,3);
  ctx.restore();
};
const drawBall=(x,y)=>{
  const b=Math.sin(tick*.22)*5;
  ctx.save();ctx.translate(Math.round(x),Math.round(y+b));
  ctx.fillStyle='rgba(0,0,0,.2)';
  ctx.beginPath();ctx.ellipse(0,14-b*.4,8,2.5,0,0,Math.PI*2);ctx.fill();
  ctx.fillStyle='#e65100';ctx.beginPath();ctx.arc(0,0,9,0,Math.PI*2);ctx.fill();
  ctx.strokeStyle='#111';ctx.lineWidth=1;
  ctx.beginPath();ctx.moveTo(-9,0);ctx.lineTo(9,0);ctx.stroke();
  ctx.beginPath();ctx.arc(0,0,9,0,Math.PI);ctx.stroke();
  ctx.restore();
};
const update=()=>{
  if(over)return;
  elapsed=(Date.now()-t0)/1000;tick++;
  let dx=0,dy=0;
  if(keys['w']||keys['arrowup'])dy=-3.5;
  if(keys['s']||keys['arrowdown'])dy=3.5;
  if(keys['a']||keys['arrowleft'])dx=-3.5;
  if(keys['d']||keys['arrowright'])dx=3.5;
  if(dx&&dy){dx*=.707;dy*=.707;}
  if(dx)p.dir=dx>0?1:-1;
  if(!blocked(p.x+dx,p.y,PR))p.x+=dx;
  if(!blocked(p.x,p.y+dy,PR))p.y+=dy;
  const spd=Math.min(1.5+elapsed*.04,4);
  const ddx=p.x-l.x,ddy=p.y-l.y;
  const dist=Math.sqrt(ddx*ddx+ddy*ddy);
  if(dist>1){
    const mx=(ddx/dist)*spd,my=(ddy/dist)*spd;
    if(!blocked(l.x+mx,l.y+my,LR)){l.x+=mx;l.y+=my;}
    else{if(!blocked(l.x+mx,l.y,LR))l.x+=mx;if(!blocked(l.x,l.y+my,LR))l.y+=my;}
    l.dir=ddx>=0?1:-1;
  }
  if(dist<PR+LR+2){over=true;if(elapsed>best)best=elapsed;}
};
const render=()=>{
  ctx.clearRect(0,0,W,H);
  drawCourt();drawObs();
  drawBall(p.x+p.dir*20,p.y-10);
  drawFigure(p.x,p.y,'#e53935','11',p.dir);
  drawFigure(l.x,l.y,'#552583','23',l.dir);
  ctx.fillStyle='rgba(0,0,0,.72)';
  ctx.beginPath();ctx.roundRect(W/2-130,6,260,32,7);ctx.fill();
  ctx.textAlign='center';ctx.textBaseline='middle';
  ctx.fillStyle='#fff';ctx.font=`bold ${Math.round(W*.017)}px monospace`;
  ctx.fillText(`⏱ ${elapsed.toFixed(1)}s`,W/2,22);
  ctx.fillStyle='#fdb927';ctx.font='bold 11px monospace';
  ctx.textAlign='right';ctx.fillText(`Best: ${best.toFixed(1)}s`,W/2+124,22);
  const spd=Math.min(1.5+elapsed*.04,4);
  if(spd>2.8){
    ctx.fillStyle=`rgba(230,50,50,${Math.min((spd-2.8)*.5,.9)})`;
    ctx.font='bold 12px monospace';ctx.textAlign='center';
    ctx.fillText('⚡ LeBron is heating up!',W/2,46);
  }
  if(over){
    ctx.fillStyle='rgba(0,0,0,.75)';ctx.fillRect(0,0,W,H);
    const bw=Math.min(380,W*.75),bh=185,bx=(W-bw)/2,by=(H-bh)/2;
    ctx.fillStyle='#0d0d1a';ctx.strokeStyle='#fdb927';ctx.lineWidth=3;
    ctx.beginPath();ctx.roundRect(bx,by,bw,bh,14);ctx.fill();ctx.stroke();
    ctx.fillStyle='#fdb927';ctx.beginPath();ctx.roundRect(bx,by,bw,6,[14,14,0,0]);ctx.fill();
    ctx.textAlign='center';ctx.textBaseline='middle';
    ctx.fillStyle='#fdb927';ctx.font=`bold ${Math.round(bw*.062)}px monospace`;
    ctx.fillText('🏀 LeBron stole the ball!',W/2,by+48);
    ctx.fillStyle='#ff6f00';ctx.font=`bold ${Math.round(bw*.048)}px monospace`;
    ctx.fillText(`Survived  ${elapsed.toFixed(1)}s`,W/2,by+96);
    ctx.fillStyle='#888';ctx.font=`${Math.round(bw*.033)}px monospace`;
    ctx.fillText(`Best: ${best.toFixed(1)}s`,W/2,by+136);
    ctx.fillStyle='#ccc';ctx.fillText('Click court then press R',W/2,by+165);
  }
};
(function loop(){update();render();requestAnimationFrame(loop);})();
})();
</script>

## Reflection

Building this game was the best way to actually understand these CS111 concepts because I had to use all of them together at once. The `for` loop draws the court, the `Array` holds the obstacles, the `Object` tracks player state, `Boolean` flags control game flow, and the math operators power the physics and AI. Seeing how they all connect in a real project made it way easier to understand than just reading about them separately.